# Module 01 — Causal Identification

## Architectural Contract

This module estimates **causal price elasticity** under partial observability.

It **does not assume** the existence of pre-aggregated datasets.

Instead, it defines a strict data flow:

Raw Calendar + Listings  
→ (Audit-valid preprocessing)  
→ Weekly Causal Panel  
→ Triangulated Causal Estimation

All transformations are explicit and reproducible.


## Why Aggregation Is Required

From Module 00 (Data Audit), we established:

- Daily prices and bookings are never jointly observed
- Missing prices on booked days are structural censoring
- Price imputation would fabricate counterfactuals
- Daily causal identification is impossible

Therefore, the **minimum valid causal unit** is:

> (listing_id, week)

This module constructs that unit explicitly.


In [1]:
import sys
import os
import numpy as np
import pandas as pd
import logging

sys.path.append(os.path.abspath(".."))

from pricing_engine.data_loader import load_and_clean_seattle_data
from pricing_engine.causal_model import (
    run_causal_pipeline,
    FixedEffectElasticity,
    LinearDMLElasticity,
    CausalForestElasticity,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("Module01_CausalID")


In [2]:
from pathlib import Path



try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()

# SCRIPT_DIR is the 'notebooks' folder.
# SCRIPT_DIR.parent is the 'Dynamic Pricing Engine' folder.
PROJECT_ROOT = SCRIPT_DIR.parent

# Now build the path from the project root
CALENDAR_PATH = PROJECT_ROOT / 'data' / 'calendar.csv'
LISTINGS_PATH = PROJECT_ROOT / 'data' / 'listings.csv'



In [3]:
df_daily = load_and_clean_seattle_data(
    calendar_path=CALENDAR_PATH,
    listings_path=LISTINGS_PATH,
)

print("Daily rows:", len(df_daily))
df_daily.head()

INFO:PriceEngine_Loader:Loading raw data...
INFO:PriceEngine_Loader:Total rows loaded: 1393570
INFO:PriceEngine_Loader:Action observed rate: 67.06%
INFO:PriceEngine_Loader:Exposure rate: 67.06%
INFO:PriceEngine_Loader:Booked proxy rate: 32.94%


Daily rows: 1393570


,listing_id,date,available,price,price_raw,action_observed,exposed,is_booked_proxy,outcome_observed,dow,...,month,neighborhood,neighbourhood,latitude,longitude,room_type,property_type,accommodates,bedrooms,bathrooms
0,241032,2016-01-04,t,85.0,$85.00,True,True,0,True,0,...,1,Queen Anne,Queen Anne,47.636289,-122.371025,Entire home/apt,Apartment,4,1.0,1.0
1,241032,2016-01-05,t,85.0,$85.00,True,True,0,True,1,...,1,Queen Anne,Queen Anne,47.636289,-122.371025,Entire home/apt,Apartment,4,1.0,1.0
2,241032,2016-01-06,f,NaN,NaN,False,False,1,False,2,...,1,Queen Anne,Queen Anne,47.636289,-122.371025,Entire home/apt,Apartment,4,1.0,1.0
3,241032,2016-01-07,f,NaN,NaN,False,False,1,False,3,...,1,Queen Anne,Queen Anne,47.636289,-122.371025,Entire home/apt,Apartment,4,1.0,1.0
4,241032,2016-01-08,f,NaN,NaN,False,False,1,False,4,...,1,Queen Anne,Queen Anne,47.636289,-122.371025,Entire home/apt,Apartment,4,1.0,1.0


## Construct Weekly Causal Panel

We now explicitly construct the **weekly causal unit**.

For each (listing_id, week):

Treatment:
- avg_price = mean price on exposed days (available = true)

Outcome:
- is_booked = 1 if any booking occurs during the week

Weeks with zero exposure are discarded.
This restores joint support without fabricating prices.


In [4]:
# --------------------------------------------------
# Weekly causal aggregation (CORRECT)
# --------------------------------------------------

# Ensure datetime
df_daily["date"] = pd.to_datetime(df_daily["date"])

# Define week start
df_daily["week_date"] = df_daily["date"].dt.to_period("W").dt.start_time

# Weekly aggregation
weekly_df = (
    df_daily
    .groupby(["listing_id", "week_date"])
    .agg(
        # Treatment: mean price over exposed days only
        avg_price=("price", lambda x: x[x.notna()].mean()),

        # Outcome: any booking during the week (proxy)
        is_booked=("is_booked_proxy", "max"),

        # Diagnostics
        exposure_days=("exposed", "sum"),
        booked_days=("is_booked_proxy", "sum"),
    )
    .reset_index()
)

# Drop weeks with zero exposure (no observed price → no treatment)
weekly_df = weekly_df[weekly_df["exposure_days"] > 0]

print("Weekly panel shape:", weekly_df.shape)
weekly_df.head()


Weekly panel shape: (141080, 6)


,listing_id,week_date,avg_price,is_booked,exposure_days,booked_days
8,3335,2016-02-29,120.0,0,7,0
9,3335,2016-03-07,120.0,0,7,0
10,3335,2016-03-14,120.0,0,7,0
11,3335,2016-03-21,120.0,0,7,0
12,3335,2016-03-28,120.0,0,7,0


## Confounder Construction

We include only:
- Pre-treatment
- Slow-moving
- Non-leaking variables

No availability-based or post-booking signals are used.


In [5]:
weekly_df["week_of_year"] = weekly_df["week_date"].dt.isocalendar().week.astype(int)
weekly_df["is_weekend_heavy"] = (
    df_daily.groupby(["listing_id", "week_date"])["date"]
    .apply(lambda x: (x.dt.dayofweek >= 5).mean())
    .reindex(
        weekly_df.set_index(["listing_id", "week_date"]).index
    )
    .values
)

# Static listing attributes (example)
listing_static = (
    df_daily
    .groupby("listing_id")
    .agg(
        accommodates=("accommodates", "first"),
        bedrooms=("bedrooms", "first"),
        bathrooms=("bathrooms", "first"),
    )
    .reset_index()
)

weekly_df = weekly_df.merge(
    listing_static,
    on="listing_id",
    how="left",
)


## Structural Sanity Checks

Before causal modeling, we verify:

- Booking outcome is non-degenerate
- Prices vary across weeks
- Exposure is non-trivial


In [6]:
print("Weekly booking rate:", weekly_df["is_booked"].mean())
print("Avg price stats:")
weekly_df["avg_price"].describe()


Weekly booking rate: 0.07582931669974483
Avg price stats:


count    141080.000000
mean        137.866839
std         104.446874
min          20.000000
25%          75.000000
50%         107.857143
75%         160.446429
max        1650.000000
Name: avg_price, dtype: float64

In [7]:
confounder_cols = [
    "week_of_year",
    "is_weekend_heavy",
    "accommodates",
    "bedrooms",
    "bathrooms",
]

X = weekly_df[confounder_cols].fillna(0)


## Causal Triangulation

We now run three complementary causal models:

1. Fixed Effects OLS — falsification
2. Linear DML — production elasticity
3. Causal Forest — heterogeneity exploration

No single model is trusted in isolation.


In [8]:
results = run_causal_pipeline(
    df_weekly=weekly_df,
    feature_cols=confounder_cols,
    price_col="avg_price",
    outcome_col="is_booked",
    listing_col="listing_id",
    week_col="week_date",
)

results


INFO:PriceEngine_Causal:>>> Running Model 1: Fixed Effects OLS (Sanity Check)
INFO:PriceEngine_Causal:[FixedEffects] Elasticity = -0.0006 [-0.0007, -0.0006]
INFO:PriceEngine_Causal:>>> Running Model 2: Linear DML (Production Elasticity)
INFO:PriceEngine_Causal:[LinearDML] Fitting | n=141080 | 3-fold | 50 bootstrap
INFO:PriceEngine_Causal:[LinearDML] Elasticity = -0.0323 [-0.0351, -0.0297]
INFO:PriceEngine_Causal:>>> Running Model 3: Causal Forest (Heterogeneity / Strategy)
INFO:PriceEngine_Causal:[CausalForest] Mean elasticity = -0.0241 (std=0.0694)


{'sanity_check': CausalResult(model_name='FixedEffectsOLS', elasticity=np.float64(-0.0006043265020357493), std_err=np.float64(2.4230833445919687e-05), ci_lower=-0.0006518184703577117, ci_upper=-0.0005568345337137869, p_value=np.float64(5.389796418338666e-137), n_obs=141080, notes={'role': 'sanity_check_only'}),
 'production': CausalResult(model_name='LinearDML', elasticity=np.float64(-0.032345504399840984), std_err=np.float64(0.0015152071706531756), ci_lower=np.float64(-0.03514792982453629), ci_upper=np.float64(-0.029669261941190798), p_value=np.float64(0.0), n_obs=141080, notes={'role': 'production', 'bootstrap': 50}),
 'strategy': CausalResult(model_name='CausalForest', elasticity=-0.024131857492132217, std_err=0.06942096255886568, ci_lower=-0.17553681069768778, ci_upper=0.032869205989322636, p_value=None, n_obs=141080, notes={'role': 'heterogeneity_analysis', 'heterogeneity_std': 0.06942096255886568})}

## Interpretation Logic

We do not validate against ground truth.

Instead, we require **triangulation consistency**:

- Fixed Effects elasticity < 0
- Linear DML agrees in sign and scale
- Causal Forest mean aligns directionally

Disagreement triggers investigation, not deployment.


In [9]:
import numpy as np
import pandas as pd


class CausalGateError(RuntimeError):
    """Hard stop for causal pipeline violations."""
    pass


class CausalGatekeeper:
    """
    Enforces GO / NO-GO causal validity gates.
    Gates validate outputs — never models.
    """

    # ======================================================
    # GATE 0 — RAW DATA CONTRACT
    # ======================================================
    @staticmethod
    def gate_raw_schema(df: pd.DataFrame):
        required_cols = {
            "listing_id", "date", "price",
            "action_observed", "exposed", "is_booked_proxy"
        }
        missing = required_cols - set(df.columns)
        if missing:
            raise CausalGateError(f"Missing required columns: {missing}")

    @staticmethod
    def gate_no_price_imputation(df: pd.DataFrame):
        if df.loc[~df["action_observed"], "price"].notna().any():
            raise CausalGateError(
                "Price observed on non-action days (imputation leakage)."
            )

    # ======================================================
    # GATE 1 — DAILY-LEVEL IMPOSSIBILITY (EXPECTED)
    # ======================================================
    @staticmethod
    def gate_daily_joint_support(df: pd.DataFrame):
        joint = df[df["action_observed"] & (df["is_booked_proxy"] == 1)]
        if len(joint) > 0:
            raise CausalGateError(
                "Daily joint support detected — causal contamination."
            )

    # ======================================================
    # GATE 2 — WEEKLY AGGREGATION VIABILITY
    # ======================================================
    @staticmethod
    def gate_weekly_joint_support(weekly_df: pd.DataFrame):
        if weekly_df["avg_price"].notna().sum() == 0:
            raise CausalGateError("No observed prices in weekly panel.")

        if not weekly_df["is_booked"].isin([0, 1]).all():
            raise CausalGateError("Weekly outcome is not binary.")

    @staticmethod
    def gate_booking_rate(weekly_df: pd.DataFrame):
        rate = weekly_df["is_booked"].mean()
        if rate <= 0.01 or rate >= 0.9:
            raise CausalGateError(
                f"Degenerate booking rate: {rate:.3f}"
            )

    @staticmethod
    def gate_exposure_precedes_outcome(weekly_df: pd.DataFrame):
        if weekly_df["exposure_days"].median() < 3:
            raise CausalGateError(
                "Insufficient exposure before booking."
            )

    # ======================================================
    # GATE 3 — IDENTIFIABILITY
    # ======================================================
    @staticmethod
    def gate_price_variation(weekly_df: pd.DataFrame,
                             min_std=5.0,
                             min_share=0.3):
        price_std = weekly_df.groupby("listing_id")["avg_price"].std()
        share = (price_std > min_std).mean()
        if share < min_share:
            raise CausalGateError(
                f"Insufficient price variation: {share:.2%}"
            )

    # ======================================================
    # GATE 4 — ECONOMIC SANITY
    # ======================================================
    @staticmethod
    def gate_negative_correlation(weekly_df: pd.DataFrame):
        corr = weekly_df["avg_price"].corr(weekly_df["is_booked"])
        if np.isnan(corr) or corr >= 0:
            raise CausalGateError(
                f"Non-negative price-demand correlation: {corr}"
            )

    # ======================================================
    # GATE 5 — SANITY MODEL (FE)
    # ======================================================
    @staticmethod
    def gate_fe_sign(elasticity: float):
        if elasticity >= 0:
            raise CausalGateError(
                f"Sanity check failed: FE elasticity = {elasticity:.4f}"
            )

    # ======================================================
    # GATE 6 — PRODUCTION MODEL (NON-INVASIVE)
    # ======================================================
    @staticmethod
    def gate_ci_excludes_zero(result):
        if result.ci_lower <= 0 <= result.ci_upper:
            raise CausalGateError(
                f"CI includes zero: [{result.ci_lower}, {result.ci_upper}]"
            )

    @staticmethod
    def gate_relative_precision(result, max_ratio=1.0):
        ratio = result.std_err / abs(result.elasticity)
        if ratio > max_ratio:
            raise CausalGateError(
                f"Unstable estimate: std/|beta| = {ratio:.2f}"
            )

    @staticmethod
    def gate_min_samples(result, min_n=10_000):
        if result.n_obs < min_n:
            raise CausalGateError(
                f"Too few observations: n={result.n_obs}"
            )

    # ======================================================
    # GATE 7 — TRIANGULATION
    # ======================================================
    @staticmethod
    def gate_triangulation(results: dict):
        signs = {
            k: np.sign(v.elasticity)
            for k, v in results.items()
            if hasattr(v, "elasticity")
        }
        if len(set(signs.values())) > 1:
            raise CausalGateError(
                f"Triangulation disagreement: {signs}"
            )

    # ======================================================
    # GATE 8 — POLICY READINESS
    # ======================================================
    @staticmethod
    def gate_elasticity_bounds(elasticity: float,
                               low=-5.0,
                               high=-0.01):
        if not (low < elasticity < high):
            raise CausalGateError(
                f"Elasticity out of safe bounds: {elasticity}"
            )


In [10]:
G = CausalGatekeeper()
# Raw data
# After FE model
G.gate_fe_sign(results["sanity_check"].elasticity)

# After DML (NON-invasive)
G.gate_ci_excludes_zero(results["production"])
G.gate_relative_precision(results["production"])
G.gate_min_samples(results["production"])

# Cross-model consistency
G.gate_triangulation(results)

# Before bandit
G.gate_elasticity_bounds(results["production"].elasticity)
